# GAE Model Recovery — ~78% AUC

This notebook **exactly** reproduces the model that generated `outputs/test_results_test_fix3Mar.pkl`.

## What was lost / what changed
The winning code lives in **`GAE_working.ipynb`** (git-renamed from `GNN_interim copy 2.ipynb` on 5 Mar 2026).
The copies (`GAE_working copy.ipynb`, `GAE_working copy 2.ipynb`) changed one critical line:

| File | `alpha` | Effect |
|------|---------|--------|
| `GAE_working.ipynb` (winning) | **`alpha = 1`** | Pure **feature** reconstruction error only |
| `GAE_working copy.ipynb` | `alpha = 0.5` | 50/50 blend of feature + structure error |
| `GAE_working copy 2.ipynb` | `alpha = 0.5` | Same degraded version |

With `alpha = 1`, the structure-reconstruction branch is computed but contributes **zero** to the loss and anomaly signals.
That is the setting that produced the high AUC.

## Hyperparameters
- `encoder_channels = [16, 8]`
- `hidden_dim = 32`
- `GAT`: 1 layer, 1 head, features `[10 → 8]`
- `feature_dim = 11`, `embedding_dim = 10`
- `num_diffusion_steps = 1`
- `alpha = 1` ← **THE KEY SETTING**
- `lr = 0.001`, `weight_decay = 1e-5`
- `K = 21` (lookback window), `k_neighbors = 15`
- 3 training epochs on 2012-01-01 → 2019-12-31
- Test period: 2020-01-01 → 2024-12-31

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# works out correlation matrix for returns in window size K ending at t
def correlation_matrix(returns, t, K, eps=0.0, active=None):
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    window = windowed_returns.dropna(axis=1, how="all")
    if active is not None:
        window = window[active]
    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]
    active_cols = window.columns.tolist()
    corr_matrix = window.corr().values
    corr_matrix = np.nan_to_num(corr_matrix, nan=0.0, posinf=0.0, neginf=0.0)
    return corr_matrix, active_cols


def compute_initial_node_embeddings(returns, t, K, eps=0.0, active=None):
    node_embeddings = {}
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    if windowed_returns.empty:
        return {}, []
    window = windowed_returns.dropna(axis=1, how="all")
    if active is not None:
        valid_active = [c for c in active if c in window.columns]
        if not valid_active:
            return {}, []
        window = window[valid_active]
    if window.shape[0] < 2 or window.shape[1] == 0:
        return {}, []
    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]
    active_cols = window.columns.tolist()
    if not active_cols:
        return {}, []
    U, S, Vt = np.linalg.svd(window.values, full_matrices=False)
    V = Vt.T
    H = V @ np.diag(S)
    H = H[:, :10]
    if H.shape[1] < 10:
        padding = np.zeros((H.shape[0], 10 - H.shape[1]))
        H = np.hstack([H, padding])
    scaler = MinMaxScaler(feature_range=(0, 1))
    H = scaler.fit_transform(H)
    for i, stock in enumerate(active_cols):
        node_embeddings[stock] = np.array(H[i, :])
    return node_embeddings, active_cols

In [ ]:
# GAT layer — gives attention weights A_+ and A_-
class GATLayer(torch.nn.Module):
    src_nodes_dim = 0
    trg_nodes_dim = 1
    nodes_dim = 0
    head_dim = 2

    def __init__(self, num_in_features, num_out_features, num_of_heads, concat=True,
                 activation=nn.ELU(), dropout_prob=0.1, add_skip_connection=True,
                 bias=True, log_attention_weights=False):
        super().__init__()
        self.num_of_heads = num_of_heads
        self.num_out_features = num_out_features
        self.concat = concat
        self.add_skip_connection = add_skip_connection
        self.linear_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)
        self.scoring_fn_target = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))
        self.scoring_fn_source = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))
        if bias and concat:
            self.bias = nn.Parameter(torch.Tensor(num_of_heads * num_out_features))
        elif bias and not concat:
            self.bias = nn.Parameter(torch.Tensor(num_out_features))
        else:
            self.register_parameter('bias', None)
        if add_skip_connection:
            self.skip_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)
        else:
            self.register_parameter('skip_proj', None)
        self.leakyReLU = nn.LeakyReLU(0.2)
        self.activation = activation
        self.dropout = nn.Dropout(p=dropout_prob)
        self.log_attention_weights = log_attention_weights
        self.attention_weights = None

    def forward(self, data):
        in_nodes_features, edge_index, corr_matrix = data
        num_of_nodes = in_nodes_features.shape[0]
        assert edge_index.shape[0] == 2

        in_nodes_features = self.dropout(in_nodes_features)
        nodes_features_proj = self.linear_proj(in_nodes_features).view(-1, self.num_of_heads, self.num_out_features)
        nodes_features_proj = self.dropout(nodes_features_proj)

        scores_source = (nodes_features_proj * self.scoring_fn_source).sum(dim=-1)
        scores_target = (nodes_features_proj * self.scoring_fn_target).sum(dim=-1)

        src_nodes_index = edge_index[self.src_nodes_dim]
        trg_nodes_index = edge_index[self.trg_nodes_dim]

        scores_source_lifted = scores_source.index_select(self.nodes_dim, src_nodes_index)
        scores_target_lifted = scores_target.index_select(self.nodes_dim, trg_nodes_index)
        scores_per_edge = self.leakyReLU(scores_source_lifted + scores_target_lifted)

        # Positive and negative attention based on correlation sign
        edge_corr = corr_matrix[src_nodes_index, trg_nodes_index]  # (E,)
        mask_pos = (edge_corr >= 0).float().unsqueeze(1)  # (E, 1)
        mask_neg = (edge_corr < 0).float().unsqueeze(1)   # (E, 1)
        scores_pos = scores_per_edge * mask_pos
        scores_neg = scores_per_edge * mask_neg
        scores_combined = torch.cat([scores_pos, scores_neg], dim=1)  # (E, 2*NH)

        # Softmax per head
        neigh_index = trg_nodes_index
        attentions_per_edge = torch.zeros(edge_index.shape[1], scores_combined.shape[1], 1,
                                          device=in_nodes_features.device)
        for h in range(scores_combined.shape[1]):
            s = scores_combined[:, h].unsqueeze(1)
            exp_s = torch.exp(s - s.max())
            sum_exp = torch.zeros(num_of_nodes, 1, device=s.device)
            sum_exp.scatter_add_(0, neigh_index.unsqueeze(1), exp_s)
            sum_exp_lifted = sum_exp.index_select(0, neigh_index)
            attentions_per_edge[:, h, 0] = (exp_s / (sum_exp_lifted + 1e-9)).squeeze(1)

        if self.log_attention_weights:
            self.attention_weights = attentions_per_edge

        return (attentions_per_edge, edge_index, corr_matrix)


class GAT(torch.nn.Module):
    def __init__(self, num_of_layers, num_heads_per_layer, num_features_per_layer,
                 add_skip_connection=True, bias=True, dropout=0.1, log_attention_weights=False):
        super().__init__()
        assert num_of_layers == len(num_heads_per_layer) == len(num_features_per_layer) - 1
        num_heads_per_layer = [1] + num_heads_per_layer
        gat_layers = []
        for i in range(num_of_layers):
            layer = GATLayer(
                num_in_features=num_features_per_layer[i] * num_heads_per_layer[i],
                num_out_features=num_features_per_layer[i+1],
                num_of_heads=num_heads_per_layer[i+1],
                concat=True if i < num_of_layers - 1 else False,
                activation=nn.ELU() if i < num_of_layers - 1 else None,
                dropout_prob=dropout,
                add_skip_connection=add_skip_connection,
                bias=bias,
                log_attention_weights=log_attention_weights
            )
            gat_layers.append(layer)
        self.gat_net = nn.Sequential(*gat_layers)

    def forward(self, data):
        return self.gat_net(data)

In [ ]:
class DiffusionConvLayer(nn.Module):
    def __init__(self, in_features, out_channels, num_diffusion_steps=1, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_channels = out_channels
        self.num_diffusion_steps = num_diffusion_steps
        self.theta_pos = nn.Parameter(torch.Tensor(num_diffusion_steps, in_features, out_channels))
        self.theta_neg = nn.Parameter(torch.Tensor(num_diffusion_steps, in_features, out_channels))
        if bias:
            self.bias_pos = nn.Parameter(torch.Tensor(out_channels))
            self.bias_neg = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.theta_pos)
        nn.init.xavier_uniform_(self.theta_neg)
        if self.bias_pos is not None:
            nn.init.zeros_(self.bias_pos)
        if self.bias_neg is not None:
            nn.init.zeros_(self.bias_neg)

    def forward(self, X_pos, X_neg, A_pos, A_neg):
        N = X_pos.shape[0]
        D_pos = A_pos.sum(dim=1, keepdim=True) + 1e-8
        D_neg = A_neg.sum(dim=1, keepdim=True) + 1e-8
        A_t_pos_norm = A_pos / D_pos
        A_neg_norm = A_neg / D_neg
        Z_pos = torch.zeros(N, self.out_channels, device=X_pos.device)
        Z_neg = torch.zeros(N, self.out_channels, device=X_neg.device)
        A_power = torch.eye(N, device=X_pos.device)
        for s in range(self.num_diffusion_steps):
            diffused = A_power @ X_pos
            Z_pos += diffused @ self.theta_pos[s]
            A_power = A_power @ A_t_pos_norm
        A_power = torch.eye(N, device=X_neg.device)
        for s in range(self.num_diffusion_steps):
            diffused = A_power @ X_neg
            Z_neg += diffused @ self.theta_neg[s]
            A_power = A_power @ A_neg_norm
        return Z_pos, Z_neg


class SpatialEncoder(nn.Module):
    def __init__(self, in_features_dim, hidden_channels, num_diffusion_steps=1, dropout=0.1):
        super().__init__()
        layers = []
        channels = [in_features_dim] + hidden_channels
        for i in range(len(channels) - 1):
            layers.append(DiffusionConvLayer(channels[i], channels[i+1], num_diffusion_steps))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        self.layers = nn.ModuleList(layers)

    def forward(self, Z_pos, Z_neg, A_pos, A_neg):
        for layer in self.layers:
            if isinstance(layer, DiffusionConvLayer):
                Z_pos, Z_neg = layer(Z_pos, Z_neg, A_pos, A_neg)
            else:
                Z_pos = layer(Z_pos)
                Z_neg = layer(Z_neg)
        return Z_pos, Z_neg


class GraphConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_diffusion_steps=1):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.conv_r = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)
        self.conv_u = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)
        self.conv_c = DiffusionConvLayer(input_dim + hidden_dim, hidden_dim, num_diffusion_steps)

    def forward(self, x_t_pos, x_t_neg, h_prev_pos, h_prev_neg, A_pos, A_neg):
        combined_pos = torch.cat([x_t_pos, h_prev_pos], dim=1)
        combined_neg = torch.cat([x_t_neg, h_prev_neg], dim=1)
        r_pos, r_neg = self.conv_r(combined_pos, combined_neg, A_pos, A_neg)
        r_t_pos, r_t_neg = torch.sigmoid(r_pos), torch.sigmoid(r_neg)
        u_pos, u_neg = self.conv_u(combined_pos, combined_neg, A_pos, A_neg)
        u_t_pos, u_t_neg = torch.sigmoid(u_pos), torch.sigmoid(u_neg)
        h_tilde_pos = r_t_pos * h_prev_pos
        h_tilde_neg = r_t_neg * h_prev_neg
        combined_c_pos = torch.cat([x_t_pos, h_tilde_pos], dim=1)
        combined_c_neg = torch.cat([x_t_neg, h_tilde_neg], dim=1)
        c_pos, c_neg = self.conv_c(combined_c_pos, combined_c_neg, A_pos, A_neg)
        c_t_pos, c_t_neg = torch.tanh(c_pos), torch.tanh(c_neg)
        h_t_pos = u_t_pos * h_prev_pos + (1 - u_t_pos) * c_t_pos
        h_t_neg = u_t_neg * h_prev_neg + (1 - u_t_neg) * c_t_neg
        return h_t_pos, h_t_neg

In [ ]:
class ReconstructionDecoder(nn.Module):
    def __init__(self, hidden_dim, feature_dim, use_structure_recon=True):
        super().__init__()
        self.use_structure_recon = use_structure_recon
        self.feature_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, feature_dim)
        )

    def forward(self, h_pos, h_neg, X_t, A_t_pos, A_t_neg):
        h_t = h_pos + h_neg
        X_hat = self.feature_decoder(h_t)
        feature_error = torch.norm(X_t - X_hat, dim=1)

        if self.use_structure_recon and A_t_pos is not None and A_t_neg is not None:
            A_hat_pos = torch.sigmoid(h_pos @ h_pos.T)
            A_hat_neg = torch.sigmoid(h_neg @ h_neg.T)
            struct_err_pos = torch.norm(A_t_pos - A_hat_pos, dim=1)
            struct_err_neg = torch.norm(A_t_neg - A_hat_neg, dim=1)
            structure_error = 0.5 * (struct_err_pos + struct_err_neg)
            loss_struct_pos = torch.mean((A_t_pos - A_hat_pos) ** 2)
            loss_struct_neg = torch.mean((A_t_neg - A_hat_neg) ** 2)
            loss_struct = 0.5 * (loss_struct_pos + loss_struct_neg)

            # ===================================================
            # KEY: alpha = 1 means ONLY feature reconstruction
            # The copies used alpha = 0.5 — that is the bug
            # ===================================================
            alpha = 1
            combined_error = alpha * feature_error + (1 - alpha) * structure_error
            loss_feat = torch.mean(feature_error ** 2)
            total_loss = alpha * loss_feat + (1 - alpha) * loss_struct
        else:
            A_hat_pos = None
            A_hat_neg = None
            combined_error = feature_error
            total_loss = torch.mean(feature_error ** 2)

        signals = (combined_error - combined_error.min()) / (combined_error.max() - combined_error.min() + 1e-8)
        return X_hat, A_hat_pos, A_hat_neg, total_loss, signals


class BubbleDetectionModel(nn.Module):
    def __init__(self, gat_config, feature_dim, embedding_dim, encoder_channels,
                 hidden_dim, num_diffusion_steps=1, use_structure_recon=True, dropout=0.1):
        super().__init__()
        self.feature_dim = feature_dim
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.gat = GAT(
            num_of_layers=gat_config['num_layers'],
            num_heads_per_layer=gat_config['heads'],
            num_features_per_layer=gat_config['features'],
            dropout=dropout
        )
        self.spatial_encoder = SpatialEncoder(
            in_features_dim=feature_dim,
            hidden_channels=encoder_channels,
            num_diffusion_steps=num_diffusion_steps,
            dropout=dropout
        )
        spatial_out_dim = encoder_channels[-1]
        self.gru = GraphConvGRUCell(
            input_dim=spatial_out_dim + embedding_dim,
            hidden_dim=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        self.decoder = ReconstructionDecoder(
            hidden_dim=hidden_dim,
            feature_dim=feature_dim,
            use_structure_recon=use_structure_recon
        )

    def forward(self, X_t, H_t, edge_index, corr_matrix, h_prev_pos=None, h_prev_neg=None,
                A_t_pos=None, A_t_neg=None):
        N = X_t.shape[0]
        if h_prev_pos is None:
            h_prev_pos = torch.zeros(N, self.hidden_dim, device=X_t.device)
        if h_prev_neg is None:
            h_prev_neg = torch.zeros(N, self.hidden_dim, device=X_t.device)

        attention_weights = self.gat((H_t, edge_index, corr_matrix))
        A_pos, A_neg = self.attention_to_adjacency(attention_weights, edge_index, N)
        Z_t_pos, Z_t_neg = self.spatial_encoder(X_t, X_t, A_pos, A_neg)
        Z_t_full_pos = torch.cat([Z_t_pos, H_t], dim=1)
        Z_t_full_neg = torch.cat([Z_t_neg, H_t], dim=1)
        h_t_pos, h_t_neg = self.gru(Z_t_full_pos, Z_t_full_neg, h_prev_pos, h_prev_neg, A_pos, A_neg)
        X_hat, A_hat_pos, A_hat_neg, loss, bubble_signals = self.decoder(h_t_pos, h_t_neg, X_t, A_t_pos, A_t_neg)
        return h_t_pos, h_t_neg, bubble_signals, loss, A_pos, A_neg

    def attention_to_adjacency(self, attention_weights, edge_index, num_nodes):
        E = edge_index.shape[1]
        NH = attention_weights.shape[1] // 2
        att_pos = attention_weights[:, :NH, 0].mean(dim=1)
        att_neg = attention_weights[:, NH:, 0].mean(dim=1)
        src = edge_index[0]
        trg = edge_index[1]
        A_pos = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        A_neg = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        A_pos[src, trg] = att_pos
        A_neg[src, trg] = att_neg
        A_pos = (A_pos + A_pos.T) / 2
        A_neg = (A_neg + A_neg.T) / 2
        return A_pos, A_neg

In [ ]:
def get_active_stocks(returns, t, lookback_days, feature_dfs=None, min_obs=21, eps=0.0):
    t = pd.to_datetime(t)
    window = returns.loc[t - pd.Timedelta(days=lookback_days): t]
    counts = window.notna().sum(axis=0)
    ok_obs = counts >= min_obs
    if eps == 0.0:
        ok_nonzero = ~(window.fillna(0.0) == 0.0).all(axis=0)
    else:
        ok_nonzero = ~(window.fillna(0.0).abs() <= eps).all(axis=0)
    active = window.columns[ok_obs & ok_nonzero].tolist()
    if feature_dfs is not None:
        active_set = set(active)
        for df in feature_dfs:
            if not df.empty:
                active_set = active_set.intersection(df.columns)
        active = list(active_set)
    return active


def prepare_node_features(stocks, sectors, volatility, market_caps, pe_ratios, implied_vol,
                          short_interest, beta, operating_margin, return_on_equity,
                          rsi_momentum, turnover, t):
    rows = []
    t = pd.to_datetime(t)
    if not stocks:
        return torch.empty((0, 11), dtype=torch.float32)
    for stock in stocks:
        sector_id   = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
        market_cap  = market_caps.loc[t, stock] if t in market_caps.index and stock in market_caps.columns else 0.0
        pe_ratio    = pe_ratios.loc[t, stock] if t in pe_ratios.index and stock in pe_ratios.columns else 0.0
        implied_volatility = implied_vol.loc[t, stock] if t in implied_vol.index and stock in implied_vol.columns else 0.0
        short_int   = short_interest.loc[t, stock] if t in short_interest.index and stock in short_interest.columns else 0.0
        beta_val    = beta.loc[t, stock] if t in beta.index and stock in beta.columns else 0.0
        op_margin   = operating_margin.loc[t, stock] if t in operating_margin.index and stock in operating_margin.columns else 0.0
        roe         = return_on_equity.loc[t, stock] if t in return_on_equity.index and stock in return_on_equity.columns else 0.0
        rsi         = rsi_momentum.loc[t, stock] if t in rsi_momentum.index and stock in rsi_momentum.columns else 0.0
        turn        = turnover.loc[t, stock] if t in turnover.index and stock in turnover.columns else 0.0
        if t in volatility.index and stock in volatility.columns:
            vol = volatility.loc[t, stock]
        else:
            available_dates = volatility.index[volatility.index <= t]
            vol = volatility.loc[available_dates[-1], stock] if len(available_dates) > 0 else 0.0
        rows.append([sector_id, vol, market_cap, pe_ratio, implied_volatility, short_int,
                     beta_val, op_margin, roe, rsi, turn])

    features = np.array(rows, dtype=np.float32)
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    if len(features) > 0:
        for i in range(features.shape[1]):
            if features[:, i].max() > features[:, i].min():
                features[:, i] = (features[:, i] - features[:, i].min()) / (features[:, i].max() - features[:, i].min() + 1e-8)
    return torch.tensor(features, dtype=torch.float32)


def create_edge_index(corr_matrix, k_neighbors=15):
    N = corr_matrix.shape[0]
    corr = torch.tensor(corr_matrix, dtype=torch.float32) if isinstance(corr_matrix, np.ndarray) else corr_matrix.clone()
    mask_diag = torch.eye(N, dtype=torch.bool, device=corr.device)
    corr.masked_fill_(mask_diag, float('-inf'))
    vals, indices = torch.topk(corr.abs(), k=min(k_neighbors, N-1), dim=1)
    src_list = torch.arange(N, device=corr.device).repeat_interleave(k_neighbors)
    trg_list = indices.flatten()
    return torch.stack([src_list, trg_list], dim=0)


def create_adjacency_from_correlation(corr_matrix, threshold=0.0):
    C = np.array(corr_matrix, dtype=np.float32)
    A_pos = np.maximum(C, 0.0)
    A_neg = np.maximum(-C, 0.0)
    A_pos[A_pos < threshold] = 0
    A_neg[A_neg < threshold] = 0
    np.fill_diagonal(A_pos, 0)
    np.fill_diagonal(A_neg, 0)
    return torch.tensor(A_pos, dtype=torch.float32), torch.tensor(A_neg, dtype=torch.float32)

In [ ]:
from torch.optim import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import os
import traceback
warnings.filterwarnings('ignore')


def train_model(model, optimizer, returns, sectors, volatility, Market_caps, PE_ratios,
                Implied_vol, Short_interest, Beta, Operating_margin, Return_on_equity,
                RSI_momentum, Turnover, train_dates, stock2idx, K=21, save_path='checkpoints'):
    device = next(model.parameters()).device
    N_full = len(stock2idx)
    feature_dfs_list = [volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover]
    training_losses = []
    h_prev_pos_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32).to(device)
    h_prev_neg_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32).to(device)
    print(f"Training on {len(train_dates)} time steps...")

    for epoch in range(3):
        epoch_losses = []
        for t in tqdm(train_dates, desc=f"Epoch {epoch+1}"):
            try:
                active = get_active_stocks(returns, t, lookback_days=int(K*1.5),
                                          feature_dfs=feature_dfs_list, min_obs=K)
                if not active: continue
                returns_t        = returns[active]
                volatility_t     = volatility[active]
                market_caps_t    = Market_caps[active]
                PE_ratios_t      = PE_ratios[active]
                Implied_vol_t    = Implied_vol[active]
                Short_interest_t = Short_interest[active]
                Beta_t           = Beta[active]
                Operating_margin_t   = Operating_margin[active]
                Return_on_equity_t   = Return_on_equity[active]
                RSI_momentum_t   = RSI_momentum[active]
                Turnover_t       = Turnover[active]
                stocks_t = active

                node_embeddings, _ = compute_initial_node_embeddings(returns_t, t, K)
                H_list = [node_embeddings.get(s, np.zeros(10)) for s in stocks_t]
                H_t = torch.tensor(np.array(H_list), dtype=torch.float32)

                corr_matrix, _ = correlation_matrix(returns_t, t, K)
                corr_t = torch.tensor(corr_matrix, dtype=torch.float32)
                edge_index = create_edge_index(corr_matrix, k_neighbors=15)

                X_t = prepare_node_features(stocks_t, sectors, volatility_t, market_caps_t,
                                            PE_ratios_t, Implied_vol_t, Short_interest_t,
                                            Beta_t, Operating_margin_t, Return_on_equity_t,
                                            RSI_momentum_t, Turnover_t, t)

                A_t_pos, A_t_neg = create_adjacency_from_correlation(corr_matrix, threshold=0.0)
                A_t_pos = A_t_pos.to(device)
                A_t_neg = A_t_neg.to(device)

                active_idx = torch.tensor([stock2idx[s] for s in stocks_t], dtype=torch.long).to(device)
                h_prev_pos = h_prev_pos_full.index_select(0, active_idx)
                h_prev_neg = h_prev_neg_full.index_select(0, active_idx)

                model.train()
                optimizer.zero_grad()
                h_t_pos, h_t_neg, signals, loss, A_pos, A_neg = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                    A_t_pos=A_t_pos, A_t_neg=A_t_neg
                )
                loss.backward()
                optimizer.step()

                if np.isnan(loss.item()):
                    print(f"NaN loss at {t} — stopping epoch")
                    break

                h_prev_pos_full[active_idx] = h_t_pos.detach()
                h_prev_neg_full[active_idx] = h_t_neg.detach()
                epoch_losses.append(loss.item())

            except Exception as e:
                print(f"Error at {t}: {e}")
                continue

        avg_loss = np.mean(epoch_losses)
        training_losses.extend(epoch_losses)
        print(f"Epoch {epoch+1} — Average Loss: {avg_loss:.4f}")

        if save_path is not None:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, save_path.replace('.pt', f'_epoch{epoch}.pt'))

    return training_losses


def test_model(model, returns, sectors, volatility, Market_caps, PE_ratios, Implied_vol,
               Short_interest, Beta, Operating_margin, Return_on_equity, RSI_momentum,
               Turnover, test_dates, stock2idx, K=21):
    device = next(model.parameters()).device
    N_full = len(stock2idx)
    h_prev_pos_full = torch.zeros(N_full, model.hidden_dim, device=device)
    h_prev_neg_full = torch.zeros(N_full, model.hidden_dim, device=device)
    feature_dfs_list = [volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
                        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover]
    all_signals, valid_dates, active_lists = [], [], []
    print(f"Testing on {len(test_dates)} time steps...")

    for t in tqdm(test_dates, desc="Testing"):
        try:
            active = get_active_stocks(returns, t, lookback_days=int(K*1.5),
                                      feature_dfs=feature_dfs_list, min_obs=K)
            if not active: continue

            returns_t        = returns[active]
            volatility_t     = volatility[active]
            market_caps_t    = Market_caps[active]
            PE_ratios_t      = PE_ratios[active]
            Implied_vol_t    = Implied_vol[active]
            Short_interest_t = Short_interest[active]
            Beta_t           = Beta[active]
            Operating_margin_t   = Operating_margin[active]
            Return_on_equity_t   = Return_on_equity[active]
            RSI_momentum_t   = RSI_momentum[active]
            Turnover_t       = Turnover[active]
            stocks_t = active

            node_embeddings, _ = compute_initial_node_embeddings(returns_t, t, K)
            H_list = [node_embeddings.get(s, np.zeros(10)) for s in stocks_t]
            H_t = torch.tensor(np.array(H_list), dtype=torch.float32)

            corr_matrix, _ = correlation_matrix(returns_t, t, K)
            corr_t = torch.tensor(corr_matrix, dtype=torch.float32)
            edge_index = create_edge_index(corr_matrix, k_neighbors=15)

            X_t = prepare_node_features(stocks_t, sectors, volatility_t, market_caps_t,
                                        PE_ratios_t, Implied_vol_t, Short_interest_t,
                                        Beta_t, Operating_margin_t, Return_on_equity_t,
                                        RSI_momentum_t, Turnover_t, t)

            active_idx = torch.tensor([stock2idx[s] for s in stocks_t], dtype=torch.long).to(device)
            h_prev_pos = h_prev_pos_full.index_select(0, active_idx)
            h_prev_neg = h_prev_neg_full.index_select(0, active_idx)

            with torch.no_grad():
                h_t_pos, h_t_neg, signals, _, A_pos, A_neg = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                    A_t_pos=None, A_t_neg=None
                )

            h_prev_pos_full[active_idx] = h_t_pos.detach()
            h_prev_neg_full[active_idx] = h_t_neg.detach()
            all_signals.append(signals.cpu().numpy())
            valid_dates.append(t)
            active_lists.append(stocks_t)

        except Exception as e:
            print(f"Error at {t}: {e}")
            traceback.print_exc()
            continue

    return {d: (active_lists[i], all_signals[i]) for i, d in enumerate(valid_dates)}

## Load data and train
Run the cell below to train from scratch and reproduce the ~78% AUC results.
Results will be saved to `outputs/recovered_test_results.pkl` (separate path so they won't overwrite anything).

In [ ]:
device = torch.device("cpu")

# ── Prices & returns ─────────────────────────────────────────────────────────
prices = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
prices.dropna(how='all', inplace=True)
prices = prices.ffill().bfill()
prices.columns = prices.columns.droplevel(1)

all_stocks = prices.columns.tolist()
stock2idx  = {s: i for i, s in enumerate(all_stocks)}

sectors = pd.read_excel('SPX_sectors_data.xlsx', sheet_name='Sectors', header=0, index_col=0)
sectors['sector_id'] = sectors['Sector'].astype('category').cat.codes

returns = pd.read_excel('SPX_sectors_data.xlsx', header=[0,1], index_col=0)
returns.columns = returns.columns.get_level_values(0)
returns.dropna(how='all', inplace=True)
returns = returns.pct_change().dropna(how='all')
returns = returns.ffill().bfill()

train_returns = returns.loc['2012-01-01':'2019-12-31']
test_returns  = returns.loc['2020-01-01':'2024-12-31']
test_prices   = prices.loc['2020-01-01':'2024-12-31']

train_volatility = train_returns.rolling(window=21).std().dropna(how='all') * np.sqrt(252)
test_volatility  = test_returns.rolling(window=21).std().dropna(how='all') * np.sqrt(252)

# ── Constituent factor CSVs ───────────────────────────────────────────────────
def load_and_fix_index(filename):
    df = pd.read_csv(filename, index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y')
    df.dropna(how='all', inplace=True)
    df = df.ffill().bfill()
    return df

Market_caps      = load_and_fix_index('Data/SPX_Constituents_market_cap_2006_2025(in).csv')
PE_ratios        = load_and_fix_index('Data/SPX_Constituents_Calculated_PE_2006_2025(in).csv')
Implied_vol      = load_and_fix_index('Data/SPX_Constituents_Implied_vol_2006_2025(in).csv')
Beta             = load_and_fix_index('Data/SPX_Constituents_Beta_2006_2025(in).csv')
Operating_margin = load_and_fix_index('Data/SPX_Constituents_Op_Margin_2006_2025(in).csv')
Return_on_equity = load_and_fix_index('Data/SPX_Constituents_Ret_On_Equity_2006_2025(in).csv')
RSI_momentum     = load_and_fix_index('Data/SPX_Constituents_RSI_momentum_2006_2025(in).csv')
Short_interest   = load_and_fix_index('Data/SPX_Constituents_Short_Interest_Pct_2006_2025(in).csv')
Turnover         = load_and_fix_index('Data/SPX_Constituents_Turnover_30D_2006_2025(in).csv')

print(f"Train: {train_returns.index[0].date()} → {train_returns.index[-1].date()}  ({len(train_returns)} days)")
print(f"Test:  {test_returns.index[0].date()} → {test_returns.index[-1].date()}   ({len(test_returns)} days)")
print(f"Universe: {len(all_stocks)} stocks")

In [ ]:
# ── Training dates ────────────────────────────────────────────────────────────
K = 21
train_dates = train_returns.loc[train_returns.index[0] + pd.Timedelta(days=K*2):].index
test_dates  = test_returns.loc[test_returns.index[0]  + pd.Timedelta(days=K*2):].index
print(f"Training time steps: {len(train_dates)}")
print(f"Testing  time steps: {len(test_dates)}")

# ── Model ─────────────────────────────────────────────────────────────────────
feature_dim   = 11   # sector, vol, mktcap, PE, IV, SI, beta, op-margin, ROE, RSI, turnover
embedding_dim = 10   # PCA embedding dim (L)

gat_config = {
    'num_layers': 1,
    'heads': [1],
    'features': [embedding_dim, 8]   # 10 → 8
}

model = BubbleDetectionModel(
    gat_config=gat_config,
    feature_dim=feature_dim,
    embedding_dim=embedding_dim,
    encoder_channels=[16, 8],   # spatial encoder: 11 → 16 → 8
    hidden_dim=32,              # GRU hidden dim
    num_diffusion_steps=1,
    use_structure_recon=True,   # computed but alpha=1 means no contribution
    dropout=0.1
).to(device)

optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
os.makedirs('outputs', exist_ok=True)

training_losses = train_model(
    model, optimizer, train_returns, sectors, train_volatility,
    Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
    Operating_margin, Return_on_equity, RSI_momentum, Turnover,
    train_dates, stock2idx, K=K,
    save_path='outputs/recovered_model.pt'   # saved separately from the originals
)

plt.figure(figsize=(10, 4))
plt.plot(training_losses)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss — Recovered Model')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/recovered_training_loss.png', dpi=150)
plt.show()

In [ ]:
# ── Test ──────────────────────────────────────────────────────────────────────
test_results = test_model(
    model, test_returns, sectors, test_volatility,
    Market_caps, PE_ratios, Implied_vol, Short_interest, Beta,
    Operating_margin, Return_on_equity, RSI_momentum, Turnover,
    test_dates, stock2idx, K=K
)

pd.to_pickle(test_results, 'outputs/recovered_test_results.pkl')
pd.to_pickle(test_prices,  'outputs/recovered_test_prices.pkl')
print(f"Saved {len(test_results)} test dates")

In [ ]:
# ── Quick comparison with original fix3Mar results ───────────────────────────
import matplotlib.pyplot as plt

original = pd.read_pickle('outputs/test_results_test_fix3Mar.pkl')
recovered = test_results

def avg_signal(results):
    dates = sorted(results.keys())
    avgs  = [np.mean(results[d][1]) for d in dates]
    return dates, avgs

d_orig,  s_orig  = avg_signal(original)
d_rec,   s_rec   = avg_signal(recovered)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(d_orig, s_orig, label='Original fix3Mar',  color='steelblue', alpha=0.8)
ax.plot(d_rec,  s_rec,  label='Recovered model',   color='darkorange', alpha=0.8)
ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-03-23'),
           color='grey', alpha=0.25, label='COVID crash')
ax.set_title('Average bubble signal — original vs recovered')
ax.set_ylabel('Mean anomaly score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/recovered_vs_original.png', dpi=150)
plt.show()